<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Zero%20cost%20AI%20agent%20for%20report%20generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# This connects your Google Drive so files are saved permanently
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive connected!")
print("Your files will be saved in: /content/drive/MyDrive/report_agent/")

Mounted at /content/drive
✅ Google Drive connected!
Your files will be saved in: /content/drive/MyDrive/report_agent/


In [3]:
import os

# Create project structure in Google Drive
base_dir = '/content/drive/MyDrive/report_agent'
folders = ['input_reports', 'output', 'templates', 'chroma_db']

for folder in folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print("✅ Project folders created in your Google Drive!")
print("📁 Go to drive.google.com to see them")

✅ Project folders created in your Google Drive!
📁 Go to drive.google.com to see them


In [15]:
# Install everything needed (takes 2-3 minutes on first run)
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q chromadb sentence-transformers
!pip install -q pypdf python-docx docx2txt unstructured
!pip install -q jinja2
!pip install -q transformers accelerate bitsandbytes
!pip install -q torch

print("✅ All libraries installed!")

✅ All libraries installed!


In [5]:
from transformers import pipeline, BitsAndBytesConfig
import torch

print("🧠 Loading AI model... (first time takes 2-3 minutes to download)")

# Use a small, fast model that fits in free Colab GPU
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load the model with 4-bit quantization (makes it fit in free GPU memory)
llm_pipeline = pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto"
    },
    trust_remote_code=True
)

print("✅ AI model loaded and ready!")
print(f"Model: {model_name}")

🧠 Loading AI model... (first time takes 2-3 minutes to download)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ AI model loaded and ready!
Model: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
# Create the HTML template
template_content = """<!DOCTYPE html>
<html>
<head>
    <title>{{ report_title }}</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 800px; margin: 40px auto; padding: 20px; }
        h1 { color: #2c3e50; border-bottom: 2px solid #3498db; }
        h2 { color: #34495e; margin-top: 30px; }
        .meta { color: #7f8c8d; font-size: 0.9em; }
        ul { line-height: 1.8; }
    </style>
</head>
<body>
    <h1>{{ report_title }}</h1>
    <p class="meta"><strong>Date:</strong> {{ report_date }} | <strong>Author:</strong> {{ author }}</p>

    <h2>1. Executive Summary</h2>
    <p>{{ executive_summary }}</p>

    <h2>2. Key Findings</h2>
    <ul>
    {% for finding in key_findings %}
        <li>{{ finding }}</li>
    {% endfor %}
    </ul>

    <h2>3. Analysis</h2>
    <p>{{ analysis }}</p>

    <h2>4. Recommendations</h2>
    <p>{{ recommendations }}</p>
</body>
</html>"""

# Save template to Google Drive
template_path = '/content/drive/MyDrive/report_agent/templates/report_template.html'
with open(template_path, 'w') as f:
    f.write(template_content)

print(f"✅ Template saved to: {template_path}")

✅ Template saved to: /content/drive/MyDrive/report_agent/templates/report_template.html


In [13]:
import os

input_dir = '/content/drive/MyDrive/report_agent/input_reports'
files = os.listdir(input_dir)

print(f"📄 Found {len(files)} files in your input folder:")
for f in files:
    print(f"  - {f}")

if len(files) == 0:
    print("\n⚠️  No files found! Upload PDFs/DOCX to Google Drive first.")

📄 Found 7 files in your input folder:
  - Rift Region February 2026 Report.docx
  - Northern Region Jan Progress Report_05022026 Reviewed by Simbe.docx
  - Northern Region March Progress Report.docx
  - Northern Region Feb Progress Report.docx
  - RM-Western Region February 2026 Progress Report (Reviewed).docx
  - Rift Region March 2026 Report.docx
  - Rift Region January 2026 Report.docx


In [16]:
import os
import json
import re
from datetime import datetime

# === IMPORTS ===
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate # Corrected import path
from jinja2 import Environment, FileSystemLoader

# === CONFIGURATION ===
BASE_DIR = '/content/drive/MyDrive/report_agent'
INPUT_DIR = os.path.join(BASE_DIR, 'input_reports')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
TEMPLATE_DIR = os.path.join(BASE_DIR, 'templates')
CHROMA_DIR = os.path.join(BASE_DIR, 'chroma_db')

# === STEP 1: LOAD DOCUMENTS ===
print("📄 Step 1: Loading documents...")
documents = []
for filename in os.listdir(INPUT_DIR):
    filepath = os.path.join(INPUT_DIR, filename)
    ext = os.path.splitext(filename)[1].lower()

    try:
        if ext == '.pdf':
            docs = PyPDFLoader(filepath).load()
        elif ext == '.docx':
            docs = Docx2txtLoader(filepath).load()
        elif ext == '.txt':
            docs = TextLoader(filepath, encoding='utf-8').load()
        else:
            print(f"  ⚠️ Skipping {filename} (unsupported format)")
            continue

        documents.extend(docs)
        print(f"  ✅ Loaded {filename} ({len(docs)} pages)")
    except Exception as e:
        print(f"  ❌ Error loading {filename}: {e}")

if not documents:
    raise Exception("No documents loaded! Upload files first.")

# === STEP 2: CHUNK DOCUMENTS ===
print("\n✂️  Step 2: Chunking documents...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # Smaller for Colab's memory
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(documents)
print(f"  ✅ Created {len(chunks)} chunks")

# === STEP 3: BUILD VECTOR STORE ===
print("\n🧠 Step 3: Building vector database...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}  # Use the free GPU!
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("  ✅ Vector database ready")

# === STEP 4: HELPER FUNCTION TO GENERATE TEXT ===
def generate_text(prompt: str, max_length: int = 500) -> str:
    """Uses the loaded LLM pipeline to generate text."""
    result = llm_pipeline(
        prompt,
        max_new_tokens=max_length,
        temperature=0.3,
        do_sample=True,
        pad_token_id=llm_pipeline.tokenizer.eos_token_id
    )
    return result[0]['generated_text']

# === STEP 5: GENERATE EACH REPORT SECTION ===
print("\n⚙️  Step 4: Generating report sections...")

def get_context(query: str) -> str:
    """Retrieves relevant chunks from the vector store."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

# Define what to generate
sections = {
    "executive_summary": {
        "query": "What is the overall summary and main points of the documents?",
        "prompt_template": """You are a professional report writer. Based ONLY on the following context, write a concise executive summary (3-4 sentences).

CONTEXT:
{context}

EXECUTIVE SUMMARY:"""
    },
    "key_findings": {
        "query": "What are the main findings, data points, and important facts?",
        "prompt_template": """Based ONLY on the following context, list exactly 5 key findings as a JSON array of strings. Return ONLY the JSON array, nothing else.

CONTEXT:
{context}

JSON ARRAY:"""
    },
    "analysis": {
        "query": "What analysis, trends, or insights can be drawn from the data?",
        "prompt_template": """You are a professional analyst. Based ONLY on the following context, write a detailed analysis paragraph (5-6 sentences).

CONTEXT:
{context}

ANALYSIS:"""
    },
    "recommendations": {
        "query": "What recommendations or next steps should be taken?",
        "prompt_template": """Based ONLY on the following context, write 3-4 actionable recommendations as a paragraph.

CONTEXT:
{context}

RECOMMENDATIONS:"""
    }
}

report_data = {
    "report_title": "AI-Generated Analysis Report",
    "author": "Colab AI Agent",
    "report_date": datetime.now().strftime('%Y-%m-%d'),
    "key_findings": []
}

for section_name, config in sections.items():
    print(f"  🔄 Generating: {section_name}...")
    context = get_context(config["query"])
    prompt = config["prompt_template"].format(context=context)

    try:
        output = generate_text(prompt, max_length=600)

        # Clean up the output (remove the prompt that gets repeated)
        if "CONTEXT:" in output:
            output = output.split("CONTEXT:")[-1]
        if section_name.upper() + ":" in output:
            output = output.split(section_name.upper() + ":")[-1]

        # Special handling for key_findings (needs to be a list)
        if section_name == "key_findings":
            try:
                # Try to extract JSON array
                match = re.search(r'\[.*?\]', output, re.DOTALL)
                if match:
                    report_data[section_name] = json.loads(match.group())
                else:
                    # Fallback: split by lines
                    report_data[section_name] = [
                        line.strip('-•* ').strip()
                        for line in output.strip().split('\n')
                        if line.strip() and len(line.strip()) > 10
                    ][:5]
            except:
                report_data[section_name] = [output[:200]]
        else:
            report_data[section_name] = output.strip()

        print(f"    ✅ Done ({len(output)} chars)")
    except Exception as e:
        print(f"    ❌ Error: {e}")
        report_data[section_name] = "Generation failed."

# === STEP 6: RENDER THE HTML REPORT ===
print("\n📊 Step 5: Rendering final report...")
env = Environment(loader=FileSystemLoader(TEMPLATE_DIR))
template = env.get_template('report_template.html')
html_output = template.render(**report_data)

output_path = os.path.join(OUTPUT_DIR, f'report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_output)

print(f"\n🎉 SUCCESS! Your report is saved at:")
print(f"📄 {output_path}")
print(f"\n💡 To view it: Open Google Drive → report_agent → output → double-click the HTML file")

📄 Step 1: Loading documents...
  ✅ Loaded Rift Region February 2026 Report.docx (1 pages)
  ✅ Loaded Northern Region Jan Progress Report_05022026 Reviewed by Simbe.docx (1 pages)
  ✅ Loaded Northern Region March Progress Report.docx (1 pages)
  ✅ Loaded Northern Region Feb Progress Report.docx (1 pages)
  ✅ Loaded RM-Western Region February 2026 Progress Report (Reviewed).docx (1 pages)
  ✅ Loaded Rift Region March 2026 Report.docx (1 pages)
  ✅ Loaded Rift Region January 2026 Report.docx (1 pages)
  ✅ Loaded RM-Western Region March 2026 Progress Report.docx (1 pages)
  ✅ Loaded RM-Western Region January 2026 Progress Report (Reviewed).docx (1 pages)

✂️  Step 2: Chunking documents...
  ✅ Created 1448 chunks

🧠 Step 3: Building vector database...


/tmp/ipykernel_2533/1409819295.py:59: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

    ✅ Done (3638 chars)
  🔄 Generating: key_findings...


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ✅ Done (2953 chars)
  🔄 Generating: analysis...


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ✅ Done (3598 chars)
  🔄 Generating: recommendations...
    ✅ Done (3617 chars)

📊 Step 5: Rendering final report...

🎉 SUCCESS! Your report is saved at:
📄 /content/drive/MyDrive/report_agent/output/report_20260831_104455.html

💡 To view it: Open Google Drive → report_agent → output → double-click the HTML file
